# Export a run's generated charts for a simulator

Turns the autoregressive test charts that `03_train_lstm` saved in `<run>/eval/ar_test/` into one folder per song:

```text
outputs/runs/<model>/<run>/exports/
├── LOVE & ENERGY/
│   ├── LOVE and ENERGY.ogg
│   └── LOVE & ENERGY.tja
└── ...
```

Upload the whole `exports/` folder to a TJA player such as OpenTaiko to play every generated chart of the run. Each chart is written on its song's own BPM and OFFSET (from the split manifest), titled `<title> (AI)`, as an Oni course. Needs the raw audio under `data/raw/` (or `DONDER_DATA`).

In [ ]:
import json
import re
import shutil

import numpy as np

from donderformer.data.data_splitter import read_manifest
from donderformer.data.tja_export import write_tja
from donderformer.paths import DataPaths, outputs_dir

# Data root: DONDER_DATA env var, else <repo>/data. Must hold the raw audio of the run's split.
paths = DataPaths()
MODEL_NAME = "taikonation"
RUN = None              # folder name under outputs/runs/<model>/; None = the most recent run
VERSION = "clean"       # "clean" (back-to-back notes removed) or "raw" (straight from the model)

runs = outputs_dir() / "runs" / MODEL_NAME
RUN_DIR = runs / RUN if RUN else sorted(p for p in runs.iterdir() if (p / "eval" / "ar_test").is_dir())[-1]
config = json.loads((RUN_DIR / "config.json").read_text())
SPLIT = config["data"]["split"]
FRAME_MS = int(re.match(r"frame(\d+)ms", config["data"]["features"]).group(1))

SEQUENCE_DIR = RUN_DIR / "eval" / "ar_test"
EXPORT_DIR = RUN_DIR / "exports"
print(f"Run: {RUN_DIR.name} | split {SPLIT} | {FRAME_MS} ms frames | {VERSION} charts")

In [ ]:
def folder_name(title):
    """Song title made safe as a folder/file name on macOS, Linux and Windows."""
    return re.sub(r'[\\/:*?"<>|]+', "_", title).strip(" .") or "untitled"


key = {"clean": "clean", "raw": "generated"}[VERSION]
used, written, missing = set(), 0, []

for row in read_manifest(paths, SPLIT, "test"):
    sequences = SEQUENCE_DIR / f"{row['chart_id']}.npz"
    if not sequences.exists():
        missing.append(row["chart_id"])
        continue

    name = folder_name(row["title"])
    if name in used:                      # two test charts with the same title
        name = f"{name} ({row['chart_id']})"
    used.add(name)

    song_dir = EXPORT_DIR / name
    song_dir.mkdir(parents=True, exist_ok=True)
    audio = paths.raw / row["audio"]
    shutil.copy2(audio, song_dir / audio.name)

    with np.load(sequences) as data:
        classes = data[key]
    level = int(float(row["level"])) if row["level"] else None
    write_tja(
        song_dir / f"{name}.tja",
        classes,
        frame_ms=FRAME_MS,
        bpm=float(row["bpm"]),
        offset=float(row["offset"]),
        title=f"{row['title']} (AI)",
        wave=audio.name,
        course="Oni",
        level=level,
    )
    written += 1

print(f"Exported {written} songs to {EXPORT_DIR}")
if missing:
    print(f"{len(missing)} test charts have no generated sequence in {SEQUENCE_DIR}")